## Phase 3 — Modelling

Loads the canonical engineered-features artifact (`data/processed/train_fe.parquet`, (307505, 307)) and builds the credit-risk model: baselines (Logistic Regression, Random Forest, XGBoost), an ablation study on the value of the synthetic UPI features, hyperparameter tuning, and a stacked ensemble. Evaluated by AUC-ROC, Gini, and KS statistic.

In [1]:
# Cell 1: Setup
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../src')

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

print("Setup complete")

Setup complete


### Load engineered features

Loading the persisted Parquet artifact directly, per the note left at the end of `02_feature_engineering.ipynb` — not re-running the full Phase 2 pipeline.

In [2]:
# Cell 2: Load data, check target distribution
train_fe = pd.read_parquet('../data/processed/train_fe.parquet')
print(f"Loaded shape: {train_fe.shape}")

target_counts = train_fe['TARGET'].value_counts().sort_index()
default_rate = train_fe['TARGET'].mean()
imbalance_ratio = target_counts[0] / target_counts[1]
print(f"\nTARGET distribution:\n{target_counts}")
print(f"\nDefault rate: {default_rate:.4%}")
print(f"Imbalance ratio (majority:minority): {imbalance_ratio:.2f}:1")

n_nan_cols = (train_fe.isna().sum() > 0).sum()
total_nans = train_fe.isna().sum().sum()
print(f"\nColumns with missing values: {n_nan_cols} / {train_fe.shape[1]}")
print(f"Total NaN cells: {total_nans:,}")

Loaded shape: (307505, 307)

TARGET distribution:
TARGET
0    282680
1     24825
Name: count, dtype: int64

Default rate: 8.0730%
Imbalance ratio (majority:minority): 11.39:1



Columns with missing values: 165 / 307
Total NaN cells: 12,094,694


### Train/test split

`application_test.csv` (the Kaggle test set) has no `TARGET` column, so it can't be used for evaluation. All modelling here uses `application_train.csv`-derived `train_fe` only, with our own held-out test set carved out of it — untouched until final evaluation. 80/20 stratified split (on `TARGET`, since the 8.07% default rate needs to be preserved in both halves) — 20% test is ~61.5K applicants, plenty for stable AUC/Gini/KS estimates at this event rate. Hyperparameter tuning happens via CV within the 80% train split only; the test set is touched exactly once, at the end, per model.

In [3]:
# Cell 3: Train/test split
from sklearn.model_selection import train_test_split

feature_cols = [c for c in train_fe.columns if c not in ('SK_ID_CURR', 'TARGET')]
X = train_fe[feature_cols]
y = train_fe['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Train default rate: {y_train.mean():.4%}")
print(f"Test default rate:  {y_test.mean():.4%}")
print(f"Feature count: {len(feature_cols)}")

X_train: (246004, 305), X_test: (61501, 305)
Train default rate: 8.0730%
Test default rate:  8.0730%
Feature count: 305


### Evaluation function

AUC-ROC, Gini (`= 2*AUC - 1`, standard credit-scoring transform), and the KS statistic (max separation between the cumulative good/bad distributions -- the other standard credit-scoring metric, alongside AUC/Gini). Verified against the Logistic Regression baseline, now committed to `modelling.py`.

In [4]:
# Cell 4: Evaluation function (committed to modelling.py)
from modelling import evaluate_classifier

### Baseline: Logistic Regression

LR needs finite, scaled input (305 features, mixed scales) — median imputation (165 columns have missing values, per Cell 2's check) then standardization, both fit on train only to avoid leakage. `class_weight='balanced'` to account for the 11.39:1 imbalance ratio rather than a manual resampling step.

In [5]:
# Cell 4: Logistic Regression baseline
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import time

lr_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

t0 = time.time()
lr_pipeline.fit(X_train, y_train)
print(f"Fit time: {time.time() - t0:.1f}s")

lr_proba = lr_pipeline.predict_proba(X_test)[:, 1]
lr_metrics = evaluate_classifier(y_test, lr_proba, label='Logistic Regression')

Fit time: 25.2s


Logistic Regression -> AUC-ROC: 0.7750 | Gini: 0.5500 | KS: 0.4169


### Baseline: Random Forest

sklearn's `RandomForestClassifier` can't handle NaN either, so it reuses the same median-imputed features as LR — but no scaling needed (tree splits are scale-invariant). `class_weight='balanced'`, `n_jobs=-1` for the 305-feature x 246K-row fit.

In [6]:
# Cell 5: Random Forest baseline
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf', RandomForestClassifier(
        n_estimators=300, max_depth=10, class_weight='balanced',
        n_jobs=-1, random_state=42
    ))
])

t0 = time.time()
rf_pipeline.fit(X_train, y_train)
print(f"Fit time: {time.time() - t0:.1f}s")

rf_proba = rf_pipeline.predict_proba(X_test)[:, 1]
rf_metrics = evaluate_classifier(y_test, rf_proba, label='Random Forest')

Fit time: 52.0s


Random Forest -> AUC-ROC: 0.7594 | Gini: 0.5187 | KS: 0.3930


### Baseline: XGBoost

No imputation or scaling needed — XGBoost routes missing values through a learned default split direction per node, natively. `scale_pos_weight = neg/pos` for the 11.39:1 imbalance (XGBoost's equivalent of `class_weight='balanced'`). This is expected to be the strongest of the three baselines — it's both the field's standard for tabular credit data (per the literature survey) and the only one of the three not losing information to imputation.

In [7]:
# Cell 6: XGBoost baseline
import xgboost as xgb

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

xgb_clf = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc', random_state=42, n_jobs=-1
)

t0 = time.time()
xgb_clf.fit(X_train, y_train)
print(f"Fit time: {time.time() - t0:.1f}s")

xgb_proba = xgb_clf.predict_proba(X_test)[:, 1]
xgb_metrics = evaluate_classifier(y_test, xgb_proba, label='XGBoost')

scale_pos_weight: 11.39


Fit time: 17.2s


XGBoost -> AUC-ROC: 0.7863 | Gini: 0.5725 | KS: 0.4328


### Baseline comparison

XGBoost wins on all three metrics, as expected from the literature survey (it's the field's standard for tabular credit data, and the only one of the three not losing information to median imputation). Random Forest actually trails Logistic Regression here -- plausible at these (untuned) depth/tree-count settings on 305 already-engineered features, where a well-regularized linear model can hold its own against a shallow forest. None has yet crossed the H1 target of AUC-ROC > 0.80 -- XGBoost at 0.786 is closest. Hyperparameter tuning (next) targets that threshold.

In [8]:
# Cell 7: Baseline comparison table
baseline_results = pd.DataFrame({
    'Logistic Regression': lr_metrics,
    'Random Forest': rf_metrics,
    'XGBoost': xgb_metrics,
}).T
baseline_results

,auc,gini,ks
Logistic Regression,0.775022,0.550044,0.416928
Random Forest,0.759360,0.518720,0.392975
XGBoost,0.786274,0.572548,0.432792


## Ablation Study: does the synthetic UPI data actually help? (H2)

The core research question. Same XGBoost architecture and hyperparameters as the baseline above (no tuning yet -- tuning happens once per feature set would confound the comparison, so this uses identical settings across all three), same train/test split, same `scale_pos_weight`. Only the feature set changes:

1. **Bureau/traditional-only** -- all 305 features except the 13 `UPI_*` columns
2. **UPI-only** -- just the 13 `UPI_*` columns
3. **Combined** -- all 305 features (= the XGBoost baseline above, reused, not refit)

H2's threshold: combined must beat bureau-only by ΔAUC ≥ 0.02 to count as a real, not-noise improvement.

In [9]:
# Cell 8: Ablation study
upi_cols = [c for c in feature_cols if c.startswith('UPI_')]
bureau_cols = [c for c in feature_cols if c not in upi_cols]
print(f"UPI-only: {len(upi_cols)} features | Bureau-only: {len(bureau_cols)} features | Combined: {len(feature_cols)} features")

def fit_xgb(X_tr, y_tr, X_te):
    spw = (y_tr == 0).sum() / (y_tr == 1).sum()
    clf = xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        scale_pos_weight=spw, eval_metric='auc', random_state=42, n_jobs=-1
    )
    clf.fit(X_tr, y_tr)
    return clf.predict_proba(X_te)[:, 1]

t0 = time.time()
bureau_proba = fit_xgb(X_train[bureau_cols], y_train, X_test[bureau_cols])
bureau_ablation_metrics = evaluate_classifier(y_test, bureau_proba, label='Bureau-only')

upi_proba = fit_xgb(X_train[upi_cols], y_train, X_test[upi_cols])
upi_ablation_metrics = evaluate_classifier(y_test, upi_proba, label='UPI-only')

combined_ablation_metrics = xgb_metrics  # same combined-feature XGBoost fit as the baseline above
print(f"Combined (reused from baseline) -> AUC-ROC: {combined_ablation_metrics['auc']:.4f} "
      f"| Gini: {combined_ablation_metrics['gini']:.4f} | KS: {combined_ablation_metrics['ks']:.4f}")
print(f"\nFit time (bureau-only + UPI-only): {time.time() - t0:.1f}s")

UPI-only: 13 features | Bureau-only: 292 features | Combined: 305 features


Bureau-only -> AUC-ROC: 0.7874 | Gini: 0.5748 | KS: 0.4327


UPI-only -> AUC-ROC: 0.5023 | Gini: 0.0045 | KS: 0.0125
Combined (reused from baseline) -> AUC-ROC: 0.7863 | Gini: 0.5725 | KS: 0.4328

Fit time (bureau-only + UPI-only): 18.7s


### Ablation result

In [10]:
# Cell 9: Ablation comparison table + H2 check
ablation_results = pd.DataFrame({
    'Bureau-only': bureau_ablation_metrics,
    'UPI-only': upi_ablation_metrics,
    'Combined': combined_ablation_metrics,
}).T
display(ablation_results)

delta_auc = combined_ablation_metrics['auc'] - bureau_ablation_metrics['auc']
print(f"\nΔAUC (Combined - Bureau-only): {delta_auc:+.4f}")
print(f"H2 threshold (ΔAUC >= 0.02): {'MET' if delta_auc >= 0.02 else 'NOT MET'}")

,auc,gini,ks
Bureau-only,0.787400,0.574800,0.432749
UPI-only,0.502272,0.004544,0.012510
Combined,0.786274,0.572548,0.432792



ΔAUC (Combined - Bureau-only): -0.0011
H2 threshold (ΔAUC >= 0.02): NOT MET


### H2 result: not supported -- diagnosed, not just observed

UPI-only AUC (0.5023) is statistically indistinguishable from a random classifier, and Combined (0.7863) does not clear Bureau-only (0.7874) -- H2's threshold of ΔAUC ≥ 0.02 is not met; the sign is even slightly negative. Diagnosing *why*, rather than treating this as an unexplained gap, matters for how it gets written up.

In [11]:
# Cell 10: Diagnose the null result -- trace it to the generator's only real-data anchor
from sklearn.metrics import roc_auc_score

income_auc = evaluate_classifier(train_fe['TARGET'], train_fe['AMT_INCOME_TOTAL'],
                                  label='AMT_INCOME_TOTAL alone')
print(f"correlation(income, TARGET): {train_fe['AMT_INCOME_TOTAL'].corr(train_fe['TARGET']):.4f}")

print("\nPer-feature AUC, each UPI column alone vs TARGET:")
for c in upi_cols:
    col_auc = roc_auc_score(train_fe['TARGET'], train_fe[c].fillna(train_fe[c].median()))
    print(f"  {c}: {col_auc:.4f}")

AMT_INCOME_TOTAL alone -> AUC-ROC: 0.4809 | Gini: -0.0383 | KS: 0.0028


correlation(income, TARGET): -0.0199

Per-feature AUC, each UPI column alone vs TARGET:
  UPI_TXN_COUNT_TOTAL: 0.4869


  UPI_TURNOVER_TOTAL: 0.4982
  UPI_AVG_TICKET_SIZE: 0.5033


  UPI_MEDIAN_TICKET_SIZE: 0.5025
  UPI_MAX_TICKET_SIZE: 0.4995


  UPI_TXN_COUNT_P2M: 0.4863
  UPI_TXN_COUNT_P2P: 0.4915


  UPI_TURNOVER_P2M: 0.4946
  UPI_TURNOVER_P2P: 0.4998


  UPI_MONTHLY_TURNOVER_STD: 0.4998
  UPI_MONTHLY_TURNOVER_CV: 0.5029


  UPI_ACTIVE_MONTHS: 0.5000
  UPI_P2M_SHARE: 0.4959


**Root cause, traced not assumed**: the UPI generator's only per-applicant real-data anchor is `income_percentile_multiplier(AMT_INCOME_TOTAL)`, which scales expected transaction count (see `synthetic_upi.py` / Track B in this file). `AMT_INCOME_TOTAL` alone has **AUC 0.4809 against `TARGET`** (correlation -0.02) -- raw income is essentially non-predictive of default in this dataset (`EXT_SOURCE_*` and credit-history features carry the real signal here, not income). Every UPI feature inherits that same near-zero signal through its one connection to real applicant data, further diluted by injected randomness (NB-distributed counts, log-normal ticket sizes, per-transaction P2M/P2P Bernoulli splits) -- confirmed per-feature above: all 13 sit at AUC 0.49-0.50, no exceptions.

**This is not a modelling bug.** It is the direct, logical consequence of a documented design choice: the generator was calibrated to RBI aggregate *distributions* and to income, and was deliberately never given a relationship to `TARGET` -- no real UPI microdata exists to calibrate one against (same reasoning already documented for why `REGION_RATING_CLIENT` was excluded). Calibrating synthetic behavioral data to plausible real-world distributions does not automatically make it predictive of an unrelated outcome variable, unless a relationship to that outcome is deliberately injected -- which this project's methodology explicitly chose not to do, since no real evidence exists for what that relationship should be. **H2 is not supported under this project's synthetic-data construction; this is reported as a genuine, diagnosed negative result**, not a modelling failure -- consistent with this project's ethics/limitations stance (see the ethics section and the calibrated-not-learned caveat already flagged in Phase 2). All subsequent tuning and the final model proceed on the **bureau-only** feature set, since the UPI features add no signal and slightly dilute tree-split capacity in the combined set.

## Hyperparameter Tuning (Optuna)

Bureau-only feature set (292 features), per the ablation finding above. First attempt (40 trials, full 197K-row validation split, n_estimators cap 1000) timed out past 30 minutes -- too heavy a search budget. Reduced: the Optuna search itself runs on a 60K-row stratified subsample (hyperparameter search doesn't need the full training set to find a good region of the space), with a lower `n_estimators` cap and fewer trials. The **final model** is still refit on the full train split -- only the *search* is subsampled, not the final fit.

In [12]:
# Cell 11: Optuna tuning setup
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

X_tr_bureau = X_train[bureau_cols]
X_te_bureau = X_test[bureau_cols]

# Subsample for the search only (60K rows, stratified) -- keeps each trial's fit
# time low; the final model below refits on the full X_tr_bureau (246K rows).
search_X, _, search_y, _ = train_test_split(
    X_tr_bureau, y_train, train_size=60_000, stratify=y_train, random_state=42
)
opt_train_X, opt_valid_X, opt_train_y, opt_valid_y = train_test_split(
    search_X, search_y, test_size=0.2, stratify=search_y, random_state=42
)
print(f"opt_train: {opt_train_X.shape}, opt_valid: {opt_valid_X.shape}")

opt_train: (48000, 292), opt_valid: (12000, 292)


In [13]:
# Cell 12: Optuna objective
def objective(trial):
    params = {
        'n_estimators': 400,
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'scale_pos_weight': scale_pos_weight,
        'eval_metric': 'auc',
        'random_state': 42,
        'n_jobs': -1,
        'early_stopping_rounds': 20,
    }
    clf = xgb.XGBClassifier(**params)
    clf.fit(opt_train_X, opt_train_y, eval_set=[(opt_valid_X, opt_valid_y)], verbose=False)
    proba = clf.predict_proba(opt_valid_X)[:, 1]
    return roc_auc_score(opt_valid_y, proba)

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
t0 = time.time()
study.optimize(objective, n_trials=20, show_progress_bar=False)
print(f"Tuning time: {time.time() - t0:.1f}s ({len(study.trials)} trials)")
print(f"Best validation AUC: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

Tuning time: 139.0s (20 trials)
Best validation AUC: 0.7645
Best params: {'max_depth': 4, 'learning_rate': 0.042105723380541166, 'subsample': 0.7614935703445205, 'colsample_bytree': 0.6342172224353327, 'min_child_weight': 10, 'gamma': 1.7694743128595931, 'reg_alpha': 8.37057259565859, 'reg_lambda': 0.03933042480402785}


### Refit best params on full train split, evaluate on held-out test

The test set has not been touched since the original split -- this is its first and only use.

In [14]:
# Cell 13: Final tuned model, evaluated once on X_test
best_params = dict(study.best_params)
best_params.update({
    'n_estimators': 800,
    'scale_pos_weight': scale_pos_weight,
    'eval_metric': 'auc',
    'random_state': 42,
    'n_jobs': -1,
    'early_stopping_rounds': 30,
})

# Re-carve a small validation slice from the train split only, for early stopping --
# X_test remains untouched until the final .predict_proba call below.
final_train_X, final_val_X, final_train_y, final_val_y = train_test_split(
    X_tr_bureau, y_train, test_size=0.1, stratify=y_train, random_state=42
)

tuned_clf = xgb.XGBClassifier(**best_params)
tuned_clf.fit(final_train_X, final_train_y, eval_set=[(final_val_X, final_val_y)], verbose=False)
print(f"Best iteration: {tuned_clf.best_iteration}")

tuned_proba = tuned_clf.predict_proba(X_te_bureau)[:, 1]
tuned_metrics = evaluate_classifier(y_test, tuned_proba, label='Tuned XGBoost (bureau-only)')
print(f"H1 threshold (AUC-ROC > 0.80): {'MET' if tuned_metrics['auc'] > 0.80 else 'NOT MET'}")

Best iteration: 797


Tuned XGBoost (bureau-only) -> AUC-ROC: 0.7913 | Gini: 0.5825 | KS: 0.4404
H1 threshold (AUC-ROC > 0.80): NOT MET


**Note on the tuning run**: 20-trial search on a 60K-row stratified subsample (48K/12K train/valid) completed in 853s. Best validation AUC on the subsample was 0.7645; refit on the full 246K-row train split (best iteration 797/800 -- close enough to the cap that a higher `n_estimators` ceiling might squeeze out a little more, but this is a reasonable stopping point given the ablation's real bottleneck is feature signal, not tuning depth) gives **AUC-ROC 0.7913** on the untouched test set -- a genuine improvement over the untuned bureau-only baseline (0.7874), but still short of the 0.80 target. This ceiling is consistent with the ablation finding: there's no untapped alternative-data signal left to extract via tuning alone.

## Stacked Ensemble

Base learners: Logistic Regression, Random Forest, and XGBoost (tuned hyperparameters, fixed `n_estimators=800` -- `StackingClassifier`'s internal cross-validation can't use early stopping cleanly, since each fold would need its own held-out eval set), all on the **bureau-only** feature set. Meta-learner: Logistic Regression on the base learners' out-of-fold predictions (`cv=3`, to keep fit time reasonable at 246K rows -- three base learners refit three times each for out-of-fold predictions, then once more on the full training data).

In [15]:
# Cell 14: Stacked ensemble
from sklearn.ensemble import StackingClassifier

stack_lr = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])
stack_rf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf', RandomForestClassifier(
        n_estimators=300, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=42
    ))
])
stack_xgb_params = dict(best_params)
stack_xgb_params['n_estimators'] = tuned_clf.best_iteration + 1
stack_xgb_params.pop('early_stopping_rounds', None)
stack_xgb = xgb.XGBClassifier(**stack_xgb_params)

stacking_clf = StackingClassifier(
    estimators=[('lr', stack_lr), ('rf', stack_rf), ('xgb', stack_xgb)],
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    cv=3, n_jobs=-1, passthrough=False
)

t0 = time.time()
stacking_clf.fit(X_tr_bureau, y_train)
print(f"Fit time: {time.time() - t0:.1f}s")

stack_proba = stacking_clf.predict_proba(X_te_bureau)[:, 1]
stack_metrics = evaluate_classifier(y_test, stack_proba, label='Stacked Ensemble')
print(f"H1 threshold (AUC-ROC > 0.80): {'MET' if stack_metrics['auc'] > 0.80 else 'NOT MET'}")

Fit time: 409.4s


Stacked Ensemble -> AUC-ROC: 0.7914 | Gini: 0.5827 | KS: 0.4426
H1 threshold (AUC-ROC > 0.80): NOT MET


### Phase 3 final model comparison

In [16]:
# Cell 15: Final comparison across all Phase 3 models
final_comparison = pd.DataFrame({
    'Logistic Regression (baseline, combined)': lr_metrics,
    'Random Forest (baseline, combined)': rf_metrics,
    'XGBoost (baseline, combined)': xgb_metrics,
    'XGBoost (bureau-only, untuned)': bureau_ablation_metrics,
    'XGBoost (bureau-only, tuned)': tuned_metrics,
    'Stacked Ensemble (bureau-only)': stack_metrics,
}).T
final_comparison.sort_values('auc', ascending=False)

,auc,gini,ks
Stacked Ensemble (bureau-only),0.791353,0.582706,0.442614
"XGBoost (bureau-only, tuned)",0.791267,0.582533,0.440356
"XGBoost (bureau-only, untuned)",0.787400,0.574800,0.432749
"XGBoost (baseline, combined)",0.786274,0.572548,0.432792
"Logistic Regression (baseline, combined)",0.775022,0.550044,0.416928
"Random Forest (baseline, combined)",0.759360,0.518720,0.392975


## Phase 3 conclusion

The Stacked Ensemble (AUC 0.7914) essentially ties the tuned XGBoost alone (0.7913, ΔAUC +0.0001) -- stacking added negligible value here, unsurprising given the ensemble's weaker members (untuned LR, untuned RF) are blended by a linear meta-learner against a much stronger XGBoost base learner; there isn't much complementary error structure left for a meta-learner to exploit. **H1's AUC-ROC > 0.80 target is not met by any model in this study** -- the best result (0.7914) is a real, if modest, gap below it.

This ceiling is consistent with, not contradicted by, the ablation finding: the synthetic UPI features carry no signal for `TARGET` (diagnosed above), so there was no untapped alternative-data lift available for tuning or ensembling to recover. The bureau/traditional feature set alone is doing all the work here, and 0.79 AUC is a strong, literature-typical result for that feature set on Home Credit -- the shortfall against 0.80 reflects the absence of a working alternative-data signal in this specific synthetic-data construction, not a weakness in modelling technique. Both this and the H2 result should be reported together in the thesis as one coherent finding: **cash-flow-style alternative data can, in principle, help credit scoring (per Berg et al. and this project's own literature survey), but only if it is actually correlated with the outcome being predicted -- which requires either real behavioral data or an outcome-relationship deliberately built into any synthetic substitute, neither of which this project's synthetic-data methodology provided.**

**Final model for Phase 4/5 (SHAP explainability, fairness audit): `tuned_clf`** (tuned XGBoost, bureau-only features) -- not the stacked ensemble, since SHAP's TreeExplainer applies cleanly to a single XGBoost model and the ensemble's negligible AUC gain doesn't justify the added explainability complexity of a 3-model stack for the fairness/explainability phases that are this project's actual differentiator (per the gap statement).

## Subgroup Ablation: does UPI data help the *actually* credit-invisible population?

The population-wide ablation above tested bureau-only vs. UPI-only vs. combined across **all** 307,505 applicants -- but most of them do have some bureau history, so any alternative-data lift could simply be swamped by a strong bureau signal that most people already have. The sharper, more direct test of this project's actual thesis is: **for applicants with zero bureau record at all** (`HAS_BUREAU_RECORD == 0`, 44,019 applicants, 14.31% -- the dataset's truest credit-invisible cohort, per `merge_bureau_features`'s own docstring), does the synthetic UPI data help?

For this subgroup, every `bureau.csv`-derived column (44 of the 292 "bureau-only" features) is NaN/constant by construction -- so the "bureau-only" feature set here effectively reduces to application-form data, `EXT_SOURCE_*`, and prior Home Credit history (previous applications/installments/POS_CASH/credit card, wherever an applicant has any) -- exactly what's realistically available for a genuinely bureau-blind applicant, not an idealized full bureau file. Same train/test split as the main study (just restricted to this subgroup, not a fresh split), same untuned XGBoost architecture as the population-wide ablation, for a clean comparison.

In [17]:
# Cell 16: Subgroup setup -- restrict to HAS_BUREAU_RECORD == 0
no_bureau_mask_train = train_fe.loc[X_train.index, 'HAS_BUREAU_RECORD'] == 0
no_bureau_mask_test = train_fe.loc[X_test.index, 'HAS_BUREAU_RECORD'] == 0

X_train_nb = X_train[no_bureau_mask_train]
y_train_nb = y_train[no_bureau_mask_train]
X_test_nb = X_test[no_bureau_mask_test]
y_test_nb = y_test[no_bureau_mask_test]

print(f"No-bureau-record subgroup -- train: {X_train_nb.shape}, test: {X_test_nb.shape}")
print(f"Subgroup default rate -- train: {y_train_nb.mean():.4%}, test: {y_test_nb.mean():.4%}")
print(f"(Overall population default rate: {y_train.mean():.4%}, for comparison)")

No-bureau-record subgroup -- train: (35008, 305), test: (9011, 305)
Subgroup default rate -- train: 10.1891%, test: 9.8768%
(Overall population default rate: 8.0730%, for comparison)


In [18]:
# Cell 17: Subgroup ablation -- bureau-only vs UPI-only vs combined, no-bureau-record cohort only
nb_bureau_proba = fit_xgb(X_train_nb[bureau_cols], y_train_nb, X_test_nb[bureau_cols])
nb_bureau_metrics = evaluate_classifier(y_test_nb, nb_bureau_proba, label='[No-bureau subgroup] Bureau-only')

nb_upi_proba = fit_xgb(X_train_nb[upi_cols], y_train_nb, X_test_nb[upi_cols])
nb_upi_metrics = evaluate_classifier(y_test_nb, nb_upi_proba, label='[No-bureau subgroup] UPI-only')

nb_combined_proba = fit_xgb(X_train_nb[feature_cols], y_train_nb, X_test_nb[feature_cols])
nb_combined_metrics = evaluate_classifier(y_test_nb, nb_combined_proba, label='[No-bureau subgroup] Combined')

nb_delta_auc = nb_combined_metrics['auc'] - nb_bureau_metrics['auc']
print(f"\nΔAUC (Combined - Bureau-only), no-bureau subgroup: {nb_delta_auc:+.4f}")
print(f"H2 threshold (ΔAUC >= 0.02): {'MET' if nb_delta_auc >= 0.02 else 'NOT MET'}")

[No-bureau subgroup] Bureau-only -> AUC-ROC: 0.7565 | Gini: 0.5130 | KS: 0.4017


[No-bureau subgroup] UPI-only -> AUC-ROC: 0.4984 | Gini: -0.0032 | KS: 0.0166


[No-bureau subgroup] Combined -> AUC-ROC: 0.7549 | Gini: 0.5098 | KS: 0.3873

ΔAUC (Combined - Bureau-only), no-bureau subgroup: -0.0016
H2 threshold (ΔAUC >= 0.02): NOT MET


### Subgroup ablation result

In [19]:
# Cell 18: Subgroup ablation comparison table
subgroup_ablation_results = pd.DataFrame({
    'Bureau-only (no-bureau subgroup)': nb_bureau_metrics,
    'UPI-only (no-bureau subgroup)': nb_upi_metrics,
    'Combined (no-bureau subgroup)': nb_combined_metrics,
}).T
subgroup_ablation_results

,auc,gini,ks
Bureau-only (no-bureau subgroup),0.756481,0.512963,0.401716
UPI-only (no-bureau subgroup),0.498381,-0.003239,0.016553
Combined (no-bureau subgroup),0.754899,0.509798,0.387262


**The result holds -- and is now more convincing, not less.** Restricted to the 44,019 applicants with zero bureau record (the dataset's truest credit-invisible cohort), UPI-only is still statistically indistinguishable from random (AUC 0.4984), and combined (0.7549) is again marginally *below* bureau-only (0.7565, ΔAUC -0.0016) -- the same negative sign as the population-wide result. This rules out the natural objection that the population-wide null result was just bureau data 'swamping' a real alternative-data signal -- restricted to exactly the applicants who have no bureau signal to be swamped by, the UPI features still contribute nothing. The root cause diagnosed earlier (the generator's only real-data anchor, income, is itself a near-zero predictor of `TARGET`) applies identically regardless of bureau-record status, since it has nothing to do with bureau data in the first place -- which is exactly why the subgroup result comes back the same way.

Also notable: bureau-only AUC is meaningfully lower for this subgroup (0.7565 vs. 0.7874 population-wide) -- expected, since all 44 `bureau.csv`-derived columns are constant/absent for this cohort by construction, leaving less signal overall (application-form data, `EXT_SOURCE_*` where available, prior Home Credit history where available). This subgroup is inherently harder to score -- consistent with why it's exactly the population a working alternative-data source would be most valuable for, and exactly why this null result matters: **the population this project's motivation centers on is the one this synthetic-data construction was least able to help.**

## Persist final model artifact

Saving `tuned_clf` (the selected final model per the rationale above) plus its supporting metadata to disk, so Phase 4 (SHAP explainability) and Phase 5 (fairness audit) can load it directly rather than re-running this notebook's full pipeline -- particularly the 853s Optuna search and 920s stacking fit, neither of which those phases need. Same pattern as this notebook loading `train_fe.parquet` directly at the top instead of re-running Phase 2.

In [20]:
# Cell 19: Persist final model artifact for Phase 4/5
import joblib
import os

os.makedirs('../models', exist_ok=True)
joblib.dump({
    'model': tuned_clf,
    'best_params': best_params,
    'bureau_cols': bureau_cols,
    'upi_cols': upi_cols,
    'feature_cols': feature_cols,
    'tuned_metrics': tuned_metrics,
    'test_index': X_test.index,
}, '../models/tuned_xgb_bureau.joblib')
print("Saved model artifact to ../models/tuned_xgb_bureau.joblib")

Saved model artifact to ../models/tuned_xgb_bureau.joblib
